# 03. Self-improvement reward와 benchmark 해석

목표: Ornith 공식 설명의 task reward 구조를 작은 simulation으로 이해하고, benchmark 일부만 선택해 과장된 결론을 만드는 오류를 피한다. 실제 Ornith training을 재현하는 notebook은 아니다.

## 1. Frontier difficulty

공식 설명은 rollout 성공률 `p`가 목표 frontier `p*=0.2`에 가까운 과제를 선호하는 Gaussian 형태를 예로 든다. 너무 쉽거나 전혀 풀 수 없는 과제보다 어렵지만 학습 신호가 있는 과제를 선택하려는 의도다.

In [ ]:
import math


def frontier_difficulty(success_rate: float, target: float = 0.2, sigma: float = 0.15) -> float:
    return math.exp(-((success_rate - target) ** 2) / (2 * sigma ** 2))


def task_reward(validity: float, success_rate: float, novelty: float) -> float:
    return validity * frontier_difficulty(success_rate) * novelty


candidates = [
    {"name": "invalid-hard", "validity": 0.0, "success_rate": 0.1, "novelty": 1.0},
    {"name": "frontier-novel", "validity": 1.0, "success_rate": 0.2, "novelty": 0.9},
    {"name": "too-easy", "validity": 1.0, "success_rate": 0.95, "novelty": 0.9},
    {"name": "duplicate", "validity": 1.0, "success_rate": 0.2, "novelty": 0.05},
]

scored = []
for item in candidates:
    score = task_reward(item["validity"], item["success_rate"], item["novelty"])
    scored.append((item["name"], round(score, 4)))

print(sorted(scored, key=lambda row: row[1], reverse=True))
assert max(scored, key=lambda row: row[1])[0] == "frontier-novel"
assert dict(scored)["invalid-hard"] == 0.0

곱셈 reward는 hard gate를 명확히 하지만 한 signal의 noise가 전체를 크게 흔들 수 있다. 실제 system에서는 validity evaluator의 false negative, novelty embedding의 편향, rollout 표본 수에 따른 success rate 분산을 추적해야 한다.

## 2. Benchmark 승패는 metric마다 다르다

공식 모델 카드의 일부 coding 결과를 사용해 각 비교 모델보다 Ornith가 높은 항목 수를 센다. 이것은 설명용이며 종합 품질 점수가 아니다.

In [ ]:
benchmarks = {
    "Terminal-Bench Terminus-2": {"Ornith": 46.2, "Qwen3.6-35B-A3B": 52.5, "Gemma-4-31B": 42.1},
    "SWE-bench Verified": {"Ornith": 70.6, "Qwen3.6-35B-A3B": 73.4, "Gemma-4-31B": 52.0},
    "SWE-bench Pro": {"Ornith": 47.5, "Qwen3.6-35B-A3B": 49.5, "Gemma-4-31B": 35.7},
    "NL2Repo": {"Ornith": 32.4, "Qwen3.6-35B-A3B": 29.4, "Gemma-4-31B": 15.5},
}


def count_wins(opponent: str) -> tuple[int, list[str]]:
    wins = [name for name, scores in benchmarks.items() if scores["Ornith"] > scores[opponent]]
    return len(wins), wins


for opponent in ("Qwen3.6-35B-A3B", "Gemma-4-31B"):
    print(opponent, count_wins(opponent))

assert count_wins("Qwen3.6-35B-A3B")[0] == 1
assert count_wins("Gemma-4-31B")[0] == 4

## 3. 자체 평가 계획

개인 저장소에서 다음 절차로 검증한다.

1. 최근에 해결한 issue 20개에서 정답 commit을 숨긴 평가 image를 만든다.
2. network와 Git history를 차단하고 동일한 tool budget을 준다.
3. 한국어·영어 prompt, context 16K·32K를 교차 비교한다.
4. test pass, regression, wall time, token, peak memory를 기록한다.
5. 한 번의 최고 점수 대신 여러 seed의 평균과 분산을 보고한다.
6. model이 만든 patch는 사람이 review한 뒤에만 실제 branch에 적용한다.

이 계획은 공개 benchmark의 headline을 자신의 workload에서 검증 가능한 engineering decision으로 바꾼다.